## 1. Preprocessing harvest data:
Using meter_data.py (md) functions
* Loads, cleans, standardizes, merges meter data from csvs and folders
* Optional duplication check prints
* Optional list of meters prints

<div style="color:blue">Enter input:

In [ ]:
# define the base path to parent directory ie: thumbdrive folder
basepath = '/Volumes/Untitled/MeterDataTest'
#base_path = '/Desktop/MeterDataTest'

# True if want duplication check
dedup_check = False

# True if want list of meters
m_list = False

##########################################################################

# raw harvest meter data csv
#harvest
#var
#dt

Imports:

In [ ]:
import pandas as pd
import os 
import numpy as np
import sys

sys.path.append(os.path.abspath('..'))
import harvest_kwh as md # import self defined module

Loading/Preprocessing:

In [ ]:
# check if base path exists
if not md.validate_base_path(basepath):
    print(f"Error: Path {basepath} does not exist")
    exit()

# load list of meter data dataframes from CSV files in basepath
meters_df = md.load_meter_dfs(basepath)

# combine list of all meter dataframes into one dataframe
combined_df = md.concat_meter_dfs(meters_df)

# convert dataframe to csv, this will be for processing kw
combined_df.to_csv('meter_data_raw.csv', index=False)

# check if duplicate reading exist in resulting meter data
if dedup_check:
    md.duplicate_check(combined_df)

# get list of meters (from csv)
if m_list:
    md.meter_list('processed_kwh.csv')

<div class="alert alert-warning">If just want to preprocess harvest data or process kw, stop here. Otherwise continue for processing kwh

---

## 2a. Processing harvest kwh data:
Using meter_data.py (md) functions
* Data interpolation to exact 15 minute intervals

<div style="color:blue">Enter input:

In [ ]:
##########################################################################

# processed harvest kwh data csv
#harvest
#var
#dt

Loading/Processing:

In [ ]:
# processesing kwh (interpolation)
processed_kwh = md.process_kwh(combined_df)

# convert (processed kwh/interpolated meter data) dataframe to csv
processed_kwh.to_csv('processed_kwh.csv', index=False)

---

<div class="alert alert-warning">Extra code in case for future reference bellow:

### Other:
Extra work done for Eileen for data on meters for testing and small view

In [ ]:
# list of meters and first timestamp in dataset
df = pd.read_csv('interpolated_meter_data.csv', encoding="utf-8")
print(df['meter_name'].unique())
print(df.head(1))

In [ ]:
# sample of interpolated data for admin_serv_1 2025-09-10
df = pd.read_csv('interpolated_meter_data.csv', encoding="utf-8")
df['datetime'] = pd.to_datetime(df['datetime'])
date = pd.to_datetime('2025-09-10').date()
sample_data = df[(df['datetime'].dt.date == date) & (df['meter_name'] == 'admin_serv_1_mtr')]
sample_data.to_csv('sample_data.csv', index=False)

# sample of interpolated data for biomedical_science_main_a_mtr 2025-09-10
df = pd.read_csv('interpolated_meter_data.csv', encoding='utf-8')

df['datetime'] = pd.to_datetime(df['datetime'])
date = pd.to_datetime('2025-09-10').date()
sample_data = df[(df['datetime'].dt.date == date) 
    & (df['meter_name'] == 'biomedical_science_main_a_mtr') 
    & ((df['interpolated'] == True) | ((df['datetime'].dt.minute % 15 == 0) & (df['datetime'].dt.second == 0)))
    ].drop_duplicates(subset=['datetime', 'meter_name'])

sample_data.to_csv('sample_data.csv', index=False)

In [ ]:
# get just interpolated meter data from all meters on days sept 07-09 2025
df = pd.read_csv('interpolated_meter_data.csv', encoding='utf-8')
df['datetime'] = pd.to_datetime(df['datetime'])

df = df[df['datetime'].dt.date.isin([
    pd.to_datetime('2025-09-07').date(),
    pd.to_datetime('2025-09-08').date(),
    pd.to_datetime('2025-09-09').date()])]
df = df[df['is_exact'] & True]

df.to_csv('sept07-09_kwh.csv', index=False)